# coding-agent：真实模型与编码宿主实验

本 Notebook 独立运行，按 文件工具 → Session → Skill → 手动压缩 → 自动压缩 → Runtime/资源/扩展 的顺序学习。通用循环和自定义工具协议见 [agent-core 实验](../fox_agent_core/AGENT_CORE_LAB.ipynb)。配套 [宿主架构指南](ARCHITECTURE_GUIDE.md)。

选择 Python 3.14+ 内核，先 `uv sync`，从第一格顺序执行。模型与摘要请求均真实调用服务；本次拆分清空旧输出，尚未重新执行真实请求。实验文件统一保存到项目 `.foxcode/labs/coding-agent/`，不写系统临时目录。


## 0. 环境与真实模型配置

先建立三个边界：`fox_ai` 负责请求模型，`Agent` 负责循环与状态，`AgentSession` 负责 Session、Skill 和 Compaction。

本实验统一从 `fox_ai.src` 导入，避免与 `packages.fox_ai.src` 混用而产生两套 Python 类型。更换导入方式后请重启内核。

In [19]:
import sys
import importlib.util
from pathlib import Path

if sys.version_info < (3, 14):
    raise RuntimeError("请选择项目 Python 3.14+ 内核，然后重新从第一格运行。")
required = ["openai", "anthropic", "pydantic", "jsonschema", "yaml", "IPython"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"当前内核缺少 {missing}；请在项目根目录运行 uv sync 并切换到 .venv 内核。")

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "packages/fox_agent_core/src/agent.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("请从 FoxCode 仓库内打开 Notebook。")
if str(REPO_ROOT / "packages") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "packages"))

import asyncio
import getpass
import html
import json
import os
import re
import tempfile
import time
from collections import Counter
from contextvars import ContextVar
from dataclasses import asdict
from IPython.display import HTML, Markdown, display

from fox_ai.src import (
    Model, AssistantMessage, UserMessage, ToolCall, ToolResultMessage,
    TextContent, stream_simple,
)
from fox_coding_agent.src import (
    AgentSession, AgentSessionConfig, AgentSessionRuntime, ReadTool, WriteTool, EditTool,
    SessionManager, JsonlSessionStorage, CompactionSettings,
    LoadSkillsOptions, load_skills, format_skills_for_prompt,
    estimate_tokens, estimate_context_tokens, find_cut_point, should_compact,
)
from fox_agent_core.src.harness.compaction import SUMMARIZATION_SYSTEM_PROMPT

print("Python:", sys.version.split()[0])
print("Agent 源码:", REPO_ROOT / "packages/fox_agent_core")

Python: 3.14.7
Agent 源码: c:\Users\Qin\Desktop\coding_agent\FoxCode\packages\fox_agent_core


### 配置中哪些数字代表真实服务，哪些是教学预算？

- `MODEL_ID` / `BASE_URL` 是真实请求目的地，默认与已有 DeepSeek 实验一致。
- `LAB_CONTEXT_WINDOW` 是本实验交给 Harness 的**本地上下文预算**，不宣称它等于服务商实际最大窗口。
- `OUTPUT_TOKENS` 控制单次回答的输出上限；`MAX_REAL_REQUESTS` 限制本次 Notebook 的模型请求总数，包含摘要调用。
- 本实验关闭 DeepSeek thinking，以便集中观察 Agent 架构；改成其他兼容服务时，应相应调整 `sampling_params`。

读取 `.env` 只匹配指定的一个密钥名，不打印文件内容，也不把整份环境配置传给模型。找不到时才出现隐藏输入框。

In [20]:
MODEL_ID = os.getenv("FOX_AGENT_MODEL_ID", "deepseek-flash")
BASE_URL = os.getenv("FOX_AGENT_BASE_URL", "https://api.deepseek.com")
API_KEY_ENV = "DEEPSEEK_API_KEY"
LAB_CONTEXT_WINDOW = 32768       # 本地教学预算，不是对服务商规格的声明
OUTPUT_TOKENS = 1200
MAX_REAL_REQUESTS = 32

def load_key(name):
    value = os.environ.get(name, "").strip()
    env_file = REPO_ROOT / ".env"
    if not value and env_file.is_file():
        for line in env_file.read_text(encoding="utf-8-sig").splitlines():
            left, sep, right = line.strip().removeprefix("export ").partition("=")
            if sep and left.strip() == name:
                value = right.strip()
                if len(value) >= 2 and value[0] == value[-1] and value[0] in ("'", '"'):
                    value = value[1:-1]
                else:
                    value = value.split(" #", 1)[0].strip()
                break
    return value or getpass.getpass(f"请输入 {name}（不回显）：").strip()

API_KEY = load_key(API_KEY_ENV)
if not API_KEY:
    raise RuntimeError("没有可用密钥。请设置 DEEPSEEK_API_KEY 后重跑此单元。")

model = Model(
    id=MODEL_ID, name="真实模型 · Agent 学习实验", provider="deepseek",
    api="openai-completions", base_url=BASE_URL, input=["text"],
    reasoning=False, context_window=LAB_CONTEXT_WINDOW, max_tokens=OUTPUT_TOKENS,
    compat={"supportsStrictMode": False},
)
STREAM_OPTIONS = {
    "api_key": API_KEY, "max_tokens": OUTPUT_TOKENS,
    "timeout_ms": 90000, "max_retries": 0,
    "sampling_params": {"extra_body": {"thinking": {"type": "disabled"}}},
}
print({"model": model.id, "api": model.api, "base_url": model.base_url,
       "local_context_budget": model.context_window, "key_loaded": bool(API_KEY)})

{'model': 'deepseek-flash', 'api': 'openai-completions', 'base_url': 'https://api.deepseek.com', 'local_context_budget': 32768, 'key_loaded': True}


### 一组观察辅助函数（可以折叠）

下面的代码只负责记录请求和渲染事件。`observed_stream` **始终调用真实的 `stream_simple`**；包装器不会改写模型回答，也不会替代摘要。

观察表记录三类信息：模型请求及 token 用量、Agent 生命周期事件、完成消息。不要打印 `STREAM_OPTIONS`，因为其中包含密钥。

In [21]:
PHASE = ContextVar("fox_agent_lab_phase", default="未标记")
REQUESTS = []
EVENTS = []
ATTACHED = {}

def safe_text(value):
    text = str(value).replace(API_KEY, "[REDACTED]")
    return re.sub(r"sk-[A-Za-z0-9_-]{12,}", "[REDACTED]", text)

def table(rows):
    if not rows:
        print("（没有记录）")
        return
    columns = list(rows[0])
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(safe_text(row.get(c, '')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

class MeasuredStream:
    def __init__(self, inner, row):
        self.inner, self.row = inner, row

    def __aiter__(self):
        return self.inner.__aiter__()

    async def result(self):
        message = await self.inner.result()
        usage = message.usage
        self.row.update({
            "status": message.stop_reason,
            "seconds": round(time.perf_counter() - self.row["started"], 2),
            "input": usage.input, "output": usage.output,
            "cache_read": usage.cache_read, "cache_write": usage.cache_write,
            "total": usage.total_tokens or (usage.input + usage.output + usage.cache_read + usage.cache_write),
        })
        return message

    async def aclose(self):
        await self.inner.aclose()
        if self.row["status"] == "pending":
            self.row["status"] = "closed"

def observed_stream(request_model, context, options):
    if len(REQUESTS) >= MAX_REAL_REQUESTS:
        raise RuntimeError("达到本次 Notebook 的真实请求上限；检查用量后再调整 MAX_REAL_REQUESTS。")
    row = {
        "number": len(REQUESTS) + 1, "phase": PHASE.get(),
        "kind": "summary" if context.system_prompt == SUMMARIZATION_SYSTEM_PROMPT else "agent",
        "model": request_model.id, "messages": len(context.messages),
        "estimated_messages": estimate_context_tokens(context.messages),
        "tools": [t.name for t in context.tools or []],
        "started": time.perf_counter(), "status": "pending",
    }
    REQUESTS.append(row)
    try:
        return MeasuredStream(stream_simple(request_model, context, options), row)
    except Exception:
        row["status"] = "request_setup_error"
        raise

def message_text(message):
    if isinstance(message.content, str):
        return message.content
    return "\n".join(c.text for c in message.content if isinstance(c, TextContent))

def final_text(agent_or_harness):
    last = next((m for m in reversed(agent_or_harness.state.messages)
                 if isinstance(m, AssistantMessage)), None)
    return message_text(last) if last else ""

def attach(subject, label):
    old = ATTACHED.pop(id(subject), None)
    if old:
        old()

    def on_event(event, cancel_event):
        row = {"phase": PHASE.get(), "owner": label, "type": event.type}
        if event.type.startswith("tool_execution"):
            row.update({"tool": event.tool_name, "call_id": event.tool_call_id})
        if event.type == "tool_execution_start":
            row["args"] = event.args
            print(f"\n[工具开始] {event.tool_name} {safe_text(event.args)}")
        elif event.type == "tool_execution_end":
            row["is_error"] = event.is_error
            print(f"[工具结束] {event.tool_name} / error={event.is_error}")
        elif event.type == "message_update":
            delta = event.assistant_message_event
            if delta.type == "text_delta":
                print(safe_text(delta.delta), end="", flush=True)
        elif event.type in ("turn_start", "agent_end", "compaction_start", "compaction_end", "compaction_error"):
            print(f"\n[{event.type}]")
        EVENTS.append(row)

    ATTACHED[id(subject)] = subject.subscribe(on_event)

async def run_step(subject, text, phase, *, skill=None):
    token = PHASE.set(phase)
    start = len(REQUESTS)
    try:
        operation = subject.invoke_skill(skill, text) if skill else subject.prompt(text)
        await asyncio.wait_for(operation, timeout=240)
        if subject.state.error_message:
            raise RuntimeError(safe_text(subject.state.error_message))
        print(f"\n本步骤新增真实请求：{len(REQUESTS) - start}")
        return final_text(subject)
    except Exception as exc:
        raise RuntimeError(safe_text(exc)) from None
    finally:
        PHASE.reset(token)

async def run_compaction(subject, phase):
    token = PHASE.set(phase)
    try:
        return await asyncio.wait_for(subject.compact(), timeout=180)
    except Exception as exc:
        raise RuntimeError(safe_text(exc)) from None
    finally:
        PHASE.reset(token)

def show_messages(subject, limit=14):
    messages = subject.state.messages
    rows = []
    for index, message in enumerate(messages[-limit:], max(0, len(messages) - limit)):
        calls = [c.name for c in message.content if isinstance(c, ToolCall)] if isinstance(message.content, list) else []
        rows.append({"index": index, "role": message.role,
                     "calls": ", ".join(calls), "call_id": getattr(message, "tool_call_id", ""),
                     "preview": message_text(message)[:120]})
    table(rows)

def assert_paired(messages):
    waiting = set()
    for message in messages:
        if isinstance(message, AssistantMessage):
            assert not waiting, f"上一轮仍有未配对工具调用：{waiting}"
            waiting = {c.id for c in message.content if isinstance(c, ToolCall)}
        elif isinstance(message, ToolResultMessage):
            assert message.tool_call_id in waiting, "出现没有对应调用的工具结果"
            waiting.remove(message.tool_call_id)
        elif isinstance(message, UserMessage):
            assert not waiting, "用户消息插入到了未完成的工具调用组中"
    assert not waiting, f"存在未完成工具调用：{waiting}"

## 1. Harness：文件工具和持久化一起工作

现在创建一个独立练习目录，里面只有教学配置和日志。会话 JSONL 放在目录外侧的 `sessions/`，模型只能读写内侧 `workspace/`。

教学限制通过 `before_tool_call` 实现路径检查。本实验不注册 `bash`，因此不依赖 Windows Git Bash；这段路径检查演示 Hook 的用途，不是完整的操作系统沙箱。

In [22]:
STORAGE_SCOPE = "project"  # 改为 "user" 可存入用户主目录的 .foxcode
if STORAGE_SCOPE not in {"project", "user"}:
    raise ValueError("STORAGE_SCOPE 只能是 project 或 user")
FOXCODE_DIR = (REPO_ROOT if STORAGE_SCOPE == "project" else Path.home()) / ".foxcode"
LAB_BASE = FOXCODE_DIR / "labs" / "coding-agent"
LAB_BASE.mkdir(parents=True, exist_ok=True)
LAB_ROOT = Path(tempfile.mkdtemp(prefix="run-", dir=LAB_BASE))
WORKSPACE = LAB_ROOT / "workspace"
SESSION_DIR = LAB_ROOT / "sessions"
WORKSPACE.mkdir()
SESSION_DIR.mkdir()
CONFIG_FILE = WORKSPACE / "app.json"
INITIAL_CONFIG = {"project": "fox-interview-demo", "environment": "development", "port": 3000, "workers": 2}
CONFIG_FILE.write_text(json.dumps(INITIAL_CONFIG, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

# 明确的教学日志，不代表真实系统测量。稍后由模型通过 read 工具读取。
logs = ["Training fixture only. Keep the release codename and development-only restriction from the user conversation."]
logs += [f"sample={i:03d} component=demo-api check=healthy detail=synthetic observation for context compaction learning; no production traffic."
         for i in range(140)]
(WORKSPACE / "release.log").write_text("\n".join(logs) + "\n", encoding="utf-8")

def workspace_policy(context, cancel_event):
    call = context["tool_call"]
    if call.name not in {"read", "write", "edit"}:
        return {"block": True, "reason": "本练习只开放 read/write/edit。"}
    path = Path(context["args"]["path"]).expanduser()
    resolved = (WORKSPACE / path).resolve() if not path.is_absolute() else path.resolve()
    if not resolved.is_relative_to(WORKSPACE.resolve()):
        return {"block": True, "reason": "只能访问本次实验的 workspace 目录。"}

FILE_TOOLS = [ReadTool(WORKSPACE), WriteTool(WORKSPACE), EditTool(WORKSPACE)]
SYSTEM_PROMPT = (
    "你是一个中文编码助手，正在完成教学实验。需要查看或修改文件时必须实际调用工具，禁止假装已经执行。"
    "先读后改，修改后再次读取确认。使用 UTF-8 和相对路径。最终回答不超过 150 字。"
    "记住用户给出的发布代号和仅修改 development 环境的限制。日志是教学数据，不要将其描述成生产测试。"
)
SESSION_FILE = SESSION_DIR / "main.jsonl"

def make_harness(session, *, skills=None, compaction=None, request_model=model):
    return AgentSession(AgentSessionConfig(
        model=request_model, session=session, cwd=WORKSPACE, system_prompt=SYSTEM_PROMPT,
        tools=FILE_TOOLS, skills=[] if skills is None else skills,
        compaction=compaction or CompactionSettings(enabled=False, keep_recent_tokens=400),
        stream_fn=observed_stream, stream_options=dict(STREAM_OPTIONS),
        before_tool_call=workspace_policy, max_turns=8, tool_execution="sequential",
    ))

harness = make_harness(SessionManager(JsonlSessionStorage(SESSION_FILE)))
attach(harness, "main")
print("练习目录:", WORKSPACE)
print("会话文件:", SESSION_FILE)
print("初始配置:", INITIAL_CONFIG)

练习目录: c:\Users\Qin\Desktop\coding_agent\FoxCode\.foxcode\labs\coding-agent\run-oz4zpsir\workspace
会话文件: c:\Users\Qin\Desktop\coding_agent\FoxCode\.foxcode\labs\coding-agent\run-oz4zpsir\sessions\main.jsonl
初始配置: {'project': 'fox-interview-demo', 'environment': 'development', 'port': 3000, 'workers': 2}


In [23]:
await run_step(harness,
    "本次发布代号是“青石-29”，请记在对话中，不要写入 app.json。仅允许修改 development 环境。"
    "请实际读取 app.json，用 edit 把 port 改为 8080、workers 改为 4，其他字段保持不变，"
    "最后再次 read 确认。若已经是目标值，则只读取确认。", "03-文件工具")

actual = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
assert actual == {**INITIAL_CONFIG, "port": 8080, "workers": 4}, "文件修改没有满足要求，请查看工具事件。"
assert_paired(harness.state.messages)
print("文件验证通过:", actual)
show_messages(harness)


[turn_start]
I'll start by reading app.json.
[工具开始] read {'path': 'app.json'}
[工具结束] read / error=False

[turn_start]
Now I'll make both edits.
[工具开始] edit {'path': 'app.json', 'old_text': '  "port": 3000,', 'new_text': '  "port": 8080,'}
[工具结束] edit / error=False

[工具开始] edit {'path': 'app.json', 'old_text': '  "workers": 2', 'new_text': '  "workers": 4'}
[工具结束] edit / error=False

[turn_start]

[工具开始] read {'path': 'app.json'}
[工具结束] read / error=False

[turn_start]
发布代号“青石-29”已记在对话中，未写入 app.json（该文件也无此字段）。仅改动 development 环境，port→8080、workers→4，project 与 environment 未变。已重新读取确认：port 8080、workers 4。此为教学数据。
[agent_end]

本步骤新增真实请求：4
文件验证通过: {'project': 'fox-interview-demo', 'environment': 'development', 'port': 8080, 'workers': 4}


index,role,calls,call_id,preview
0,user,,,本次发布代号是“青石-29”，请记在对话中，不要写入 app.json。仅允许修改 development 环境。请实际读取 app.json，用 edit 把 port 改为 8080、workers 改为 4，其他字段保持不变，最后再次
1,assistant,read,,I'll start by reading app.json.
2,toolResult,,call_00_VOoR0qsSqIuyE0w0BPRg8203,"1: { 2: ""project"": ""fox-interview-demo"", 3: ""environment"": ""development"", 4: ""port"": 3000, 5: ""workers"": 2 6: }"
3,assistant,"edit, edit",,Now I'll make both edits.
4,toolResult,,call_00_DK2VfNZEz22GSAdcQ74L9234,Edited C:\Users\Qin\Desktop\coding_agent\FoxCode\.foxcode\labs\coding-agent\run-oz4zpsir\workspace\app.json
5,toolResult,,call_01_mRQQo3gkPrI5ZY3BinxV9029,Edited C:\Users\Qin\Desktop\coding_agent\FoxCode\.foxcode\labs\coding-agent\run-oz4zpsir\workspace\app.json
6,assistant,read,,
7,toolResult,,call_00_AeeUJWFxxRRtjQ1Qperz7674,"1: { 2: ""project"": ""fox-interview-demo"", 3: ""environment"": ""development"", 4: ""port"": 8080, 5: ""workers"": 4 6: }"
8,assistant,,,发布代号“青石-29”已记在对话中，未写入 app.json（该文件也无此字段）。仅改动 development 环境，port→8080、workers→4，project 与 environment 未变。已重新读取确认：port 80


**观察与复盘**：文件内容由 Python 读取验证，不凭模型一句“修改成功”就判定完成。AgentSession 每收到一条完成消息就保存它，工具结果也属于正式消息；一次成功文件修改并不等于整个 run 结束，模型还会读取确认并输出说明。

## 2. Session：销毁对象的假设下，怎样继续对话？

下面重新打开 JSONL 并创建新的 Harness，不把旧对象的 `state.messages` 直接传过去。先检查重建结果，再让真实模型回答只出现在历史中的发布代号。

新对象暂时关闭所有工具，避免它从文件里寻找答案。完成后恢复工具集合，供下一章使用。主线按顺序使用新对象，不让两个 Harness 同时写同一个文件。

In [24]:
entries = harness.session.get_entries()
table([{"index": i, "id": e.id[:8], "parent": (e.parent_id or "")[:8], "type": e.type}
       for i, e in enumerate(entries[-16:], max(0, len(entries) - 16))])
before_restore = [m.model_dump(mode="json") for m in harness.state.messages]
restored = make_harness(SessionManager(JsonlSessionStorage(SESSION_FILE)), request_model=None)
attach(restored, "restored")
assert [m.model_dump(mode="json") for m in restored.state.messages] == before_restore
print("重建消息数:", len(restored.state.messages), "恢复模型:", restored.state.model.id)

restored.set_active_tools([])
answer = await run_step(restored, "只根据已有对话回答：本次发布代号是什么，修改被限制在哪个环境？不要调用工具。", "04-恢复后的真实回答")
assert "青石-29" in answer, "真实模型未正确回忆代号，请检查历史和本次响应；不要用固定答案代替模型回答。"
restored.set_active_tools(["read", "write", "edit"])

index,id,parent,type
0,ecd92d8e,,model_change
1,cd5b4b15,ecd92d8e,thinking_level_change
2,999c7847,cd5b4b15,active_tools_change
3,2404db68,999c7847,message
4,2b3e90cc,2404db68,message
5,c1bc06fe,2b3e90cc,message
6,33d29318,c1bc06fe,message
7,9c5d7715,33d29318,message
8,e30b21b6,9c5d7715,message
9,78796864,e30b21b6,message


重建消息数: 9 恢复模型: deepseek-flash

[turn_start]
本次发布代号是“青石-29”，修改仅限 development 环境。
[agent_end]

本步骤新增真实请求：1


In [25]:
# 分支操作不需要调用模型：fork 有独立 session ID，消息来自选定历史路径。
branch = restored.fork()
assert branch.session.storage.get_metadata()["id"] != restored.session.storage.get_metadata()["id"]
branch.set_active_tools([])
print("原会话工具:", [t.name for t in restored.state.tools])
print("分支会话工具:", [t.name for t in branch.state.tools])
print("分支消息数:", len(branch.state.messages))

原会话工具: ['read', 'write', 'edit']
分支会话工具: []
分支消息数: 11


**面试复盘**：Session 保存的是消息、配置与分支关系。恢复时重新构建 Agent，不会恢复正在执行的 Python 协程。`fork()` 的结果当前是独立内存会话；JSONL 的逻辑追加使用全文件原子替换实现，并不是只向文件末尾写一行。

## 3. Skill：把任务方法放进上下文，工具仍负责执行

本章在练习目录创建一个 `release-audit` 技能。技能描述告诉模型何时使用，正文说明如何读取配置和输出检查报告。

我们分别观察两条路径：

1. **发现与按需读取**：system prompt 只有目录，模型用 `read` 读取 `SKILL.md`。
2. **显式调用**：`invoke_skill()` 把技能正文直接加入用户消息，不必再通过工具加载正文。

技能中的报告格式是约定，不会自己执行代码；实际读取和写入仍通过 read/write 工具完成。

In [26]:
SKILL_DIR = WORKSPACE / ".foxcode" / "skills" / "release-audit"
SKILL_DIR.mkdir(parents=True)
SKILL_FILE = SKILL_DIR / "SKILL.md"
SKILL_FILE.write_text("""---
name: release-audit
description: 检查练习项目的开发环境发布配置，并生成结构化验收报告。
---
# 开发环境发布验收

本文中的相对文件路径，除用户指定的工作目录文件外，均相对技能目录解释。
本练习工作目录下的 app.json 是检查对象，audit.md 是报告文件。

只有用户要求实际验收时，执行以下步骤：
1. 使用 read 读取工作目录下的 app.json，禁止根据旧回复猜测内容。
2. 核对 environment=development、port=8080、workers=4；任何不符都写明。
3. 使用 write 写入工作目录下的 audit.md，不修改 app.json。
4. 报告含“检查对象”“检查结果”“注意事项”三个小节。
5. 注意事项必须说明仅检查开发环境，不能把教学日志称为生产测试。
6. 报告末尾独立写一行 AUDIT_LAB_V1，作为本技能的格式验收标记。

如果用户仅要求解释本技能，不执行文件操作，直接解释方法。
""".replace("\n    ", "\n"), encoding="utf-8")

loaded = load_skills(LoadSkillsOptions(
    cwd=str(WORKSPACE), skill_paths=[str(WORKSPACE / ".foxcode/skills")], include_defaults=False,
))
assert [s.name for s in loaded.skills] == ["release-audit"]
assert not loaded.diagnostics, [asdict(d) for d in loaded.diagnostics]
catalog = format_skills_for_prompt(loaded.skills)
assert "release-audit" in catalog and "AUDIT_LAB_V1" not in catalog
print(catalog)

# 重建宿主，使发现的技能目录进入 system prompt；沿用同一个持久会话。
skilled = make_harness(SessionManager(JsonlSessionStorage(SESSION_FILE)), skills=loaded.skills, request_model=None)
attach(skilled, "with-skill")

The following skills provide specialized instructions for specific tasks.
Read the full skill file when the task matches its description.
When a skill file references a relative path, resolve it against the skill directory (parent of SKILL.md / dirname of the path).

<available_skills>
  <skill>
    <name>release-audit</name>
    <description>检查练习项目的开发环境发布配置，并生成结构化验收报告。</description>
    <location>c:\Users\Qin\Desktop\coding_agent\FoxCode\.foxcode\labs\coding-agent\run-oz4zpsir\workspace\.foxcode\skills\release-audit\SKILL.md</location>
  </skill>
</available_skills>


In [27]:
event_start = len(EVENTS)
await run_step(skilled,
    "请使用可用技能 release-audit 实际验收开发环境配置。先用 read 加载该技能的 SKILL.md，"
    "再严格按技能读取 app.json 并写出 audit.md，最后说明结果。", "05-Skill按需读取")
phase_events = EVENTS[event_start:]
skill_reads = [e for e in phase_events if e["type"] == "tool_execution_start" and e.get("tool") == "read"
               and "SKILL.md" in e.get("args", {}).get("path", "")]
assert skill_reads, "没有观察到真实的技能文件读取；请查看工具调用与服务的工具支持。"
report_path = WORKSPACE / "audit.md"
assert report_path.is_file(), "模型没有实际写出验收报告。"
report = report_path.read_text(encoding="utf-8")
assert "AUDIT_LAB_V1" in report, "报告没有遵循技能规定的验收标记。"
assert json.loads(CONFIG_FILE.read_text(encoding="utf-8")) == actual
display(Markdown(report))


[turn_start]



[工具开始] read {'path': '.foxcode/skills/release-audit/SKILL.md'}
[工具结束] read / error=False

[turn_start]

[工具开始] read {'path': 'app.json'}
[工具结束] read / error=False

[turn_start]

[工具开始] write {'path': 'audit.md', 'content': '# 开发环境发布验收报告\n\n发布代号：青石-29（仅记录于对话，未写入 app.json）\n\n## 检查对象\n\n工作目录下的 `app.json`（当前内容经 read 实际读取）：\n\n- project: fox-interview-demo\n- environment: development\n- port: 8080\n- workers: 4\n\n## 检查结果\n\n- environment=development：符合\n- port=8080：符合\n- workers=4：符合\n- 结论：开发环境发布配置全部符合预期，无不符合项。\n\n## 注意事项\n\n- 本次验收仅针对 development 环境，未涉及其他环境。\n- 教学日志属于教学数据，不能称为生产测试。\n- 报告阶段未修改 app.json。\n\nAUDIT_LAB_V1\n'}
[工具结束] write / error=False

[turn_start]
验收完成：按技能读取 app.json，environment=development、port=8080、workers=4 全部符合，无不符合项。已写出 audit.md（含三小节与 AUDIT_LAB_V1），未改动 app.json；仅检查开发环境，教学日志不视为生产测试。
[agent_end]

本步骤新增真实请求：4


# 开发环境发布验收报告

发布代号：青石-29（仅记录于对话，未写入 app.json）

## 检查对象

工作目录下的 `app.json`（当前内容经 read 实际读取）：

- project: fox-interview-demo
- environment: development
- port: 8080
- workers: 4

## 检查结果

- environment=development：符合
- port=8080：符合
- workers=4：符合
- 结论：开发环境发布配置全部符合预期，无不符合项。

## 注意事项

- 本次验收仅针对 development 环境，未涉及其他环境。
- 教学日志属于教学数据，不能称为生产测试。
- 报告阶段未修改 app.json。

AUDIT_LAB_V1


In [28]:
await run_step(skilled,
    "这次只解释方法，不执行文件操作。请用三条要点说明这个技能检查什么，以及 Skill 为什么不等于 Tool。",
    "05-Skill显式调用", skill="release-audit")
last_user = next(m for m in reversed(skilled.state.messages) if isinstance(m, UserMessage))
assert '<skill name="release-audit"' in message_text(last_user)
assert "AUDIT_LAB_V1" in message_text(last_user)
print("显式调用的用户消息已包含技能正文；无需把所有技能全文长期放入 system prompt。")


[turn_start]
**技能检查内容（三条要点）**

1. 读取工作目录下 app.json 实际内容，核对 environment=development、port=8080、workers=4，不符处须写明。
2. 在 audit.md 写出含“检查对象／检查结果／注意事项”三小节的报告，末尾独立一行 AUDIT_LAB_V1；全程不修改 app.json。
3. 注意事项须声明仅检查开发环境，且不得把教学日志称为生产测试。

**Skill ≠ Tool**：Tool 是可调用的原子能力（read/write 等），执行具体动作；Skill 是写给模型的操作说明与流程约定，本身不执行，只指导如何组合使用工具。
[agent_end]

本步骤新增真实请求：1
显式调用的用户消息已包含技能正文；无需把所有技能全文长期放入 system prompt。


**观察与复盘**：模型写出的 `audit.md` 是执行证据，`AUDIT_LAB_V1` 是遵循技能正文的证据。目录中不包含这个标记，因此只看名称和描述不够。

`invoke_skill()` 改变的是输入组织方式，而不是绕过工具权限。Skill 也不是独立的 Agent、操作系统权限或执行脚本。

## 4. 手动压缩：缩短下一次请求，不删除原始历史

### 4.1 让真实模型读取较长的教学日志

前面已经积累了真实工具结果和多轮回答。现在用 `read` 读入较长的重复日志，使上下文增长更明显。日志文本是本 Notebook 生成的教学输入，模型的读取决定和回答仍来自真实服务。

只要求模型给出短结论，避免把完整日志再复述一次。此时自动压缩仍关闭，所以可以清楚比较手动压缩前后。

In [29]:
await run_step(skilled,
    "为了学习上下文管理，请实际 read 工作目录下的 release.log，limit=200。"
    "这是明确标注的教学数据，不代表生产测量。读取后用两句话说明其重复性，"
    "不要复述全部日志。继续保留最初的发布代号、配置结果和仅限开发环境的约束。", "06-读取长日志")
assert_paired(skilled.state.messages)
PRE_COMPACT_LEAF = skilled.session.leaf_id
before_messages = list(skilled.state.messages)
before_estimate = estimate_context_tokens(before_messages)
before_entry_count = len(skilled.session.get_entries())
cut = find_cut_point(before_messages, skilled.session_config.compaction.keep_recent_tokens)
assert cut > 0, "当前没有可总结的前缀；增加历史或减小 keep_recent_tokens 后再实验。"
table([{"messages": len(before_messages), "estimated_tokens": before_estimate,
        "cut_index": cut, "summarized_messages": cut, "retained_messages": len(before_messages) - cut}])
print("保留尾部的首条角色:", before_messages[cut].role)


[turn_start]

[工具开始] read {'path': 'release.log', 'limit': 200}
[工具结束] read / error=False

[turn_start]
已读取 release.log 前 200 行（实际返回 141 行）。重复性：除 sample 编号递增外，各行内容完全一致，均为 component=demo-api、check=healthy 的合成观测记录；文件首行说明这是训练用夹具，须保留发布代号与仅限开发环境的约束。发布代号仍为“青石-29”，app.json 仍为 port 8080、workers 4，改动仅限 development 环境；该日志为教学数据，不属生产测试。
[agent_end]

本步骤新增真实请求：2


messages,estimated_tokens,cut_index,summarized_messages,retained_messages
25,6854,24,24,1


保留尾部的首条角色: assistant


### 4.2 由真实模型生成摘要，再提交压缩条目

`enabled=False` 只关闭**自动触发**，不禁止手动 `compact()`。切割点会避开正常的工具调用/结果配对边界。

本单元将实际调用模型生成摘要；没有注入自定义 `summary_fn`。只有摘要成功并且非空，Harness 才写入 compaction 条目并重建上下文。

In [30]:
request_start = len(REQUESTS)
compact_result = await run_compaction(skilled, "06-真实摘要手动压缩")
summary_requests = [r for r in REQUESTS[request_start:] if r["kind"] == "summary"]
assert len(summary_requests) == 1, "没有观察到一次真实摘要请求。"
assert compact_result.removed_count > 0 and compact_result.summary.strip()
assert len(skilled.session.get_entries()) == before_entry_count + 1
assert skilled.session.get_entries()[-1].type == "compaction"
assert_paired(skilled.state.messages)

after_estimate = estimate_context_tokens(skilled.state.messages)
table([
    {"阶段": "压缩前", "上下文消息": len(before_messages), "估算token": before_estimate, "历史条目": before_entry_count},
    {"阶段": "压缩后", "上下文消息": len(skilled.state.messages), "估算token": after_estimate,
     "历史条目": len(skilled.session.get_entries())},
])
print("真实摘要（不是预设答案）：")
display(Markdown(compact_result.summary))
if after_estimate >= before_estimate:
    print("本次摘要没有变短：摘要属于模型输出，压缩率不是硬保证。可缩短摘要或增加待总结内容。")


[compaction_start]

[compaction_end]


阶段,上下文消息,估算token,历史条目
压缩前,25,6854,30
压缩后,2,880,31


真实摘要（不是预设答案）：


# 对话摘要

## 背景与用户目标
用户在练习 Agent 的上下文管理与技能（Skill）机制。工作目录为 `C:\Users\Qin\Desktop\coding_agent\FoxCode\.foxcode\labs\coding-agent\run-oz4zpsir\workspace`。

## 关键约束（持续保留）
- 发布代号：**"青石-29"**（仅记在对话中，不写入 app.json；app.json 也无此字段）
- 仅允许修改 **development** 环境
- `release.log` 为**教学数据**，不代表生产测量，不得称为"生产测试"

## 已完成工作

**1. 配置修改（app.json）**
- 实际 read 了 app.json，初始值：project=fox-interview-demo, environment=development, port=3000, workers=2
- 用 edit 将 port 改为 8080、workers 改为 4（其他字段未动）
- 再次 read 确认：port=8080、workers=4、environment=development、project 未变

**2. 技能验收（release-audit）**
- read 加载技能 SKILL.md（路径：`.foxcode/skills/release-audit/SKILL.md`），并按技能要求执行
- 严格 read app.json，核对 environment=development、port=8080、workers=4，全部符合，无不符合项
- write 生成 audit.md（含"检查对象""检查结果""注意事项"三小节，末尾独立一行 AUDIT_LAB_V1），未修改 app.json

**3. 技能解释（仅解释，不执行文件操作）**
- 三条要点：① 读取 app.json 核对 environment/port/workers；② 写 audit.md 报告且不改 app.json；③ 注意事项声明仅检查开发环境、教学日志非生产测试
- 说明 Skill ≠ Tool：Tool 是可调用的原子能力（read/write），执行具体动作；Skill 是指导模型如何组合使用工具的操作说明

**4. 读取 release.log（上下文管理学习）**
- read release.log，limit=200，共读到 141 行
- 内容高度重复：第 1 行为说明行，其余均为 `sample=NNN component=demo-api check=healthy detail=synthetic...; no production traffic` 的合成记录
- 用户要求用两句话说明重复性（未复述全部日志）——**此步的最终两句话说明在对话中尚未由 assistant 给出**（对话在读到 toolResult 后中断）

## 待办 / 未完成
- **用户最后一轮要求尚未回应**：需用两句话说明 release.log 的重复性（不要复述全部日志），并继续保留发布代号、配置结果、仅限开发环境的约束。

## 报告文件
- audit.md 已写入，649 bytes，含 AUDIT_LAB_V1 标记

### 4.3 再次从 JSONL 恢复，让真实模型检验保留信息

我们不直接继续使用压缩后的 Python 对象，而是重新读磁盘。这样同时验证 **压缩结果是否保存** 和 **模型能否依据摘要继续回答**。

暂时禁用工具，防止模型通过重新读取 app.json 补答案。下面的问题不再重复给出发布代号或配置数值。

In [31]:
compact_restored = make_harness(SessionManager(JsonlSessionStorage(SESSION_FILE)), skills=loaded.skills, request_model=None)
attach(compact_restored, "after-compaction")
assert len(compact_restored.state.messages) == len(skilled.state.messages)
assert message_text(compact_restored.state.messages[0]).startswith("[Previous conversation summary]")
compact_restored.set_active_tools([])
answer_after_compaction = await run_step(compact_restored,
    "只根据已有上下文，不读取任何文件：本次发布代号是什么？当前端口和 workers 数量是什么？"
    "只允许修改哪个环境？说明教学日志是否能证明生产环境健康。请简短回答。", "06-压缩后恢复与回答")
assert "青石-29" in answer_after_compaction and "8080" in answer_after_compaction, (
    "真实模型未保留全部关键事实，这是摘要质量问题。请检查摘要并调整策略，不要硬编码回答。")
compact_restored.set_active_tools(["read", "write", "edit"])


[turn_start]
发布代号：青石-29。端口 8080，workers 4。仅允许修改 development 环境。release.log 是教学合成数据，不能证明生产环境健康。
[agent_end]

本步骤新增真实请求：1


**面试复盘**：压缩后上下文条数通常变少，历史条目反而增加一条。Session 是历史，模型 Context 是历史的一个工作视图。摘要质量并不由非空校验保证，需要像上面这样检查后续任务是否仍能正确完成。

## 5. 自动压缩：在每次模型请求前处理预算

手动压缩回答的是“我现在要求整理历史”。自动压缩回答的是“下一次请求之前，输入是否已经接近预算”。

下面从**手动压缩之前的 leaf** 分出独立会话，恢复那段较长的真实历史。然后降低给 Harness 的本地预算，使 `prepare_request` 在首次模型调用前触发压缩。

**这个预算仅在本地触发策略，不修改真实 API 服务的上下文上限。** 不伪造服务端溢出、不生成固定摘要，也不影响主会话。自动压缩应在本次 `prompt()` 内产生“summary 请求 → agent 请求”的顺序。

In [32]:
auto_session = compact_restored.session.fork(PRE_COMPACT_LEAF)
auto_history = auto_session.build_context()
history_estimate = estimate_context_tokens(auto_history)
assert history_estimate > OUTPUT_TOKENS, "当前历史过短，不能稳定展示自动压缩。"
AUTO_WINDOW = history_estimate + OUTPUT_TOKENS
AUTO_RESERVE = OUTPUT_TOKENS + 512
auto_model = model.model_copy(update={"context_window": AUTO_WINDOW})
auto_settings = CompactionSettings(enabled=True, reserve_tokens=AUTO_RESERVE, keep_recent_tokens=400)
auto = make_harness(auto_session, skills=loaded.skills, compaction=auto_settings, request_model=auto_model)
attach(auto, "auto-compaction")
auto.set_active_tools([])  # 本章只问历史，禁用工具以便观察纯粹的准备请求流程。

# 与 Harness 相同的估算组成：消息 + system prompt + 工具声明（这里是空数组）。
estimated_input = history_estimate + estimate_tokens(UserMessage(content=auto.state.system_prompt))
estimated_input += estimate_tokens(UserMessage(content=json.dumps([])))
threshold = AUTO_WINDOW - min(AUTO_RESERVE, AUTO_WINDOW // 2)
assert should_compact(estimated_input, AUTO_WINDOW, auto_settings)
table([{"历史估算": history_estimate, "含system的估算": estimated_input,
        "本地窗口": AUTO_WINDOW, "触发阈值": threshold, "保留预算": auto_settings.keep_recent_tokens}])

历史估算,含system的估算,本地窗口,触发阈值,保留预算
6854,7185,8054,6342,400


In [33]:
request_start, event_start = len(REQUESTS), len(EVENTS)
answer_auto = await run_step(auto,
    "只根据对话历史简短回答：本次发布代号、已修改的端口和 workers 数量，以及环境限制。不要读取文件。",
    "07-自动压缩与回答")
auto_requests = REQUESTS[request_start:]
auto_events = EVENTS[event_start:]
assert [r["kind"] for r in auto_requests] == ["summary", "agent"], "请检查自动压缩是否先于普通模型请求。"
assert any(e["type"] == "compaction_end" for e in auto_events)
assert "青石-29" in answer_auto and "8080" in answer_auto
table([{k: r.get(k) for k in ["number", "phase", "kind", "messages", "status", "seconds", "total"]}
       for r in auto_requests])
print("原始主会话仍可恢复；自动压缩只改变这个分支的上下文。")


[turn_start]

[compaction_start]



[compaction_end]
发布代号：青石-29。端口改为 8080，workers 改为 4。改动仅限 development 环境，代号不写入 app.json；日志为教学数据。
[agent_end]

本步骤新增真实请求：2


number,phase,kind,messages,status,seconds,total
15,07-自动压缩与回答,summary,1,stop,3.33,6495
16,07-自动压缩与回答,agent,3,stop,0.86,1096


原始主会话仍可恢复；自动压缩只改变这个分支的上下文。


**为什么是 prepare_request？** 首轮恢复出的历史就可能很长；后续 steering/follow-up 还会增加输入。每次请求前、待处理消息注入后的检查点，才能看到即将发送的内容。

**本地预算是估算，不是服务端 tokenizer。** 若摘要或最近消息仍太大，当前 Harness 会在触发压缩后的检查路径报预算错误。不要把继续降低阈值误认为能解决任意长输入；应考虑工具输出截断、分块摘要或减少输入。

## 6. Runtime：资源、扩展与新的宿主生命周期

前面的实验直接组装 Harness，便于理解内部策略；应用入口应使用 Runtime。下面在同一练习 workspace 中创建新会话，自动加载项目 Skill、AGENTS.md 与 Prompt，并注册一个只读扩展命令。历史不会自动从前面的 Harness 合并到新会话。

这里为教学只启用 read/write/edit，仍通过前面的路径 Hook 限制操作；真实 CLI 默认工具集合更大。新增的回答仍调用真实模型。


In [34]:
(WORKSPACE / "AGENTS.md").write_text("这是教学项目；请用中文解释，只处理开发环境。", encoding="utf-8")
prompt_dir = WORKSPACE / ".foxcode" / "prompts"
prompt_dir.mkdir(parents=True, exist_ok=True)
(prompt_dir / "explain.md").write_text("请用两句话解释 $ARGUMENTS，不操作文件。", encoding="utf-8")
extension_events = []

def lab_extension(api):
    api.register_command("lab-info", lambda arguments, context: {
        "cwd": str(context.cwd), "entries": len(context.session.get_entries())
    }, "观察当前宿主")
    api.add_prompt_guideline("这是架构教学实验；准确区分 Skill 与 Tool。")
    api.on("session_start", lambda data, context: extension_events.append("start"))

runtime = AgentSessionRuntime(
    WORKSPACE, model=model, user_dir=LAB_ROOT / "user-config",
    settings_overrides={"tools": ["read", "write", "edit"], "compaction": {"enabled": False}},
    stream_fn=observed_stream, stream_options=dict(STREAM_OPTIONS),
    before_tool_call=workspace_policy, extension_factories=(lab_extension,),
)
attach(runtime, "runtime")
try:
    assert "release-audit" in [skill.name for skill in runtime.resources.skills]
    assert "explain" in runtime.resources.prompts
    assert "这是教学项目" in runtime.state.system_prompt
    print(await runtime.run_command("lab-info"))
    await asyncio.wait_for(runtime.invoke_prompt("explain", "Runtime 与 Agent 的职责"), timeout=240)
    if runtime.state.error_message:
        raise RuntimeError(safe_text(runtime.state.error_message))
    before_reload = len(runtime.state.messages)
    await runtime.reload()
    assert len(runtime.state.messages) == before_reload
    print("重载保留消息:", before_reload, "扩展生命周期:", extension_events)
    print("Runtime 会话:", runtime.session_file)
finally:
    await runtime.close()


{'cwd': 'C:\\Users\\Qin\\Desktop\\coding_agent\\FoxCode\\.foxcode\\labs\\coding-agent\\run-oz4zpsir\\workspace', 'entries': 4}

[turn_start]
Runtime（运行时）负责"怎么跑"：它管理对话循环、调用模型、解析并执行工具调用、维护上下文与消息历史，是执行引擎和基础设施——在本项目里就是 FoxCode 这个宿主程序本身。

Agent（智能体）负责"做什么、按什么规矩做"：它由提示词、指令和可用技能（Skill）构成的策略来决定行为，比如本仓库的 AGENTS.md、release-audit 技能——它们向 Runtime 提供任务目标和操作规范，本身不能执行任何操作。

需要区分的是：Skill 是喂给 Agent 的指令性知识（文本），不是可调用的执行单元；Tool（如 read/write/edit）才是由 Runtime 实际执行的函数。
[agent_end]
重载保留消息: 2 扩展生命周期: ['start']
Runtime 会话: C:\Users\Qin\Desktop\coding_agent\FoxCode\.foxcode\labs\coding-agent\run-oz4zpsir\workspace\.foxcode\sessions\fd8a227758414f118ffa01c4b8820103.jsonl


## 7. 用量、证据和面试复盘

本表将普通回答请求和摘要请求分开计数。费用依赖服务端实际费率与缓存策略，本 Notebook 不用可能过期的价格估算金额；请结合服务商账单查看真实费用。

`total` 优先使用 provider 报告的 `total_tokens`，没有时才根据 fox_ai 的 input/output/cache 字段相加。前文“估算 token”与这里的实际 usage 是两个不同概念。

In [35]:
table([{k: r.get(k, "") for k in ["number", "phase", "kind", "messages", "status", "seconds", "input", "output", "total"]}
       for r in REQUESTS])
counts = Counter(r["kind"] for r in REQUESTS)
print("真实请求次数:", len(REQUESTS), "普通响应:", counts["agent"], "摘要:", counts["summary"])
print("provider 报告的累计 token:", sum(r.get("total", 0) for r in REQUESTS))

# 只保存脱敏的观测数据；不保存 options、密钥或 HTTP headers。
metrics_path = LAB_ROOT / "request_metrics.json"
public_rows = [{k: v for k, v in r.items() if k != "started"} for r in REQUESTS]
metrics_path.write_text(json.dumps(public_rows, ensure_ascii=False, indent=2), encoding="utf-8")
assert API_KEY not in metrics_path.read_text(encoding="utf-8")
assert API_KEY not in SESSION_FILE.read_text(encoding="utf-8")
print("会话与报告保留在:", LAB_ROOT)
print("请求统计:", metrics_path)

number,phase,kind,messages,status,seconds,input,output,total
1,03-文件工具,agent,1,toolUse,0.56,691,44,735
2,03-文件工具,agent,3,toolUse,1.06,154,163,957
3,03-文件工具,agent,6,toolUse,0.86,157,37,1090
4,03-文件工具,agent,8,stop,1.48,253,66,1215
5,04-恢复后的真实回答,agent,10,stop,0.77,742,17,759
6,05-Skill按需读取,agent,12,toolUse,1.48,1458,50,1508
7,05-Skill按需读取,agent,14,toolUse,1.19,381,37,1826
8,05-Skill按需读取,agent,16,toolUse,1.49,221,219,2104
9,05-Skill按需读取,agent,18,stop,1.62,240,62,2222
10,05-Skill显式调用,agent,20,stop,1.24,382,146,2704


真实请求次数: 17 普通响应: 15 摘要: 2
provider 报告的累计 token: 40739
会话与报告保留在: c:\Users\Qin\Desktop\coding_agent\FoxCode\.foxcode\labs\coding-agent\run-oz4zpsir
请求统计: c:\Users\Qin\Desktop\coding_agent\FoxCode\.foxcode\labs\coding-agent\run-oz4zpsir\request_metrics.json


### 面试复盘

| 问题 | 本实验中的证据 |
| --- | --- |
| 文件真的修改了吗？ | read/edit/read 的消息与 Python 文件断言 |
| Session 恢复了什么？ | 重新打开 JSONL 后重建的历史与模型回答 |
| Skill 等于工具吗？ | 技能目录、正文标记与实际 read/write 调用 |
| 压缩是否删除历史？ | compaction 条目增加，模型工作上下文缩短 |
| 自动压缩何时执行？ | compaction 事件先于模型请求 |
| Runtime 怎样减少入口重复？ | 统一发现资源、注册扩展、重载并保留消息 |

CLI 中同样可用 `fox --interactive`、`/compact`、`/reload` 和 `--resume`。Notebook 为方便观察关闭了部分默认策略，不应直接当作完整 CLI 的配置。
